In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np



In [6]:
# --- 1. Load the dataset ---
df = pd.read_csv('metrics.csv')
print("--- Initial Data Exploration ---")
print("First 5 rows of the dataset:")
print(df.head())
print("\nInformation about the dataset:")
print(df.info())
print("\nDescriptive statistics of the dataset:")
print(df.describe())

# --- 2. Preprocess the data ---
print("\n--- Data Preprocessing ---")
# Calculate the percentage of missing values for each column
missing_percentages = df.isnull().sum() / len(df)
print("\nPercentage of missing values per column:")
print(missing_percentages)

# Identify columns to drop (more than 50% missing values)
columns_to_drop = missing_percentages[missing_percentages > 0.50].index
print(f"\nColumns to drop (more than 50% missing): {list(columns_to_drop)}")
df_cleaned = df.drop(columns=columns_to_drop)


--- Initial Data Exploration ---
First 5 rows of the dataset:
   Year  Tobacco Price\nIndex  Retail Prices\nIndex  \
0  2015                1294.3                 386.7   
1  2014                1226.0                 383.0   
2  2013                1139.3                 374.2   
3  2012                1057.8                 363.1   
4  2011                 974.9                 351.9   

   Tobacco Price Index Relative to Retail Price Index  \
0                                              334.7    
1                                              320.1    
2                                              304.5    
3                                              291.3    
4                                              277.1    

   Real Households' Disposable Income  Affordability of Tobacco Index  \
0                               196.4                            58.7   
1                               190.0                            59.4   
2                               190.3        

In [7]:
# Impute missing values in 'Sex' column with the mode
if 'Sex' in df_cleaned.columns:
    mode_sex = df_cleaned['Sex'].mode()[0]
    df_cleaned['Sex'].fillna(mode_sex, inplace=True)
    print(f"\nMissing values in 'Sex' column imputed with mode: {mode_sex}")

# Impute remaining numerical missing values with the median
for column in df_cleaned.select_dtypes(include=['float64', 'int64']).columns:
    if df_cleaned[column].isnull().sum() > 0:
        median_value = df_cleaned[column].median()
        df_cleaned[column].fillna(median_value, inplace=True)
        print(f"Missing values in '{column}' imputed with median: {median_value}")

print("\nMissing values after imputation:")
print(df_cleaned.isnull().sum())
print("\nFirst 5 rows of the cleaned dataset:")
print(df_cleaned.head())
print("\nData types of the cleaned dataset:")
print(df_cleaned.info())



Missing values in 'Household Expenditure on Tobacco' imputed with median: 14047.0
Missing values in 'Household Expenditure Total' imputed with median: 639405.0
Missing values in 'Expenditure on Tobacco as a Percentage of Expenditure' imputed with median: 2.2

Missing values after imputation:
Year                                                     0
Tobacco Price\nIndex                                     0
Retail Prices\nIndex                                     0
Tobacco Price Index Relative to Retail Price Index       0
Real Households' Disposable Income                       0
Affordability of Tobacco Index                           0
Household Expenditure on Tobacco                         0
Household Expenditure Total                              0
Expenditure on Tobacco as a Percentage of Expenditure    0
dtype: int64

First 5 rows of the cleaned dataset:
   Year  Tobacco Price\nIndex  Retail Prices\nIndex  \
0  2015                1294.3                 386.7   
1  2014        

C:\Users\Admin\AppData\Local\Temp\ipykernel_11076\53883308.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[column].fillna(median_value, inplace=True)


In [11]:
# --- 3. Define Target Variable and Features ---
# Target variable is 'Expenditure on Tobacco as a Percentage of Expenditure' (regression)
target_column = 'Expenditure on Tobacco as a Percentage of Expenditure'
X = df_cleaned.drop(target_column, axis=1)
y = df_cleaned[target_column]

# Identify categorical and numerical features for preprocessing pipeline
# Only select features that are present in the dataframe
categorical_features = [col for col in categorical_features if col in X.columns]
numerical_features = [col for col in numerical_features if col in X.columns]
# Select numerical columns from X
# Make sure 'year' is included as a numerical feature if it exists in X
if 'Year' in numerical_features:
    numerical_features.remove('Year') # Remove temporarily to add it explicitly after one-hot encoding if needed
    numerical_features.insert(0, 'Year') # Add 'year' to the beginning of numerical features for clarity

print(f"\nCategorical features selected: {categorical_features}")
print(f"Numerical features selected: {numerical_features}")

# Create a column transformer for preprocessing: scaling numerical features and one-hot encoding categorical
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # Keep any other columns that were not transformed
)



Categorical features selected: []
Numerical features selected: ['Year', 'Tobacco Price\nIndex', 'Retail Prices\nIndex', 'Tobacco Price Index Relative to Retail Price Index', "Real Households' Disposable Income", 'Affordability of Tobacco Index', 'Household Expenditure on Tobacco', 'Household Expenditure Total']


In [12]:
# --- 4. Split the Data ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(df_cleaned.columns.tolist())
print(X.columns.tolist())

# --- 5. Model Selection, Training, and Evaluation ---
print("\n--- Model Training and Evaluation ---")
# Create a pipeline with preprocessing and RandomForestRegressor model
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', RandomForestRegressor(random_state=42))])

# Train the model
model_pipeline.fit(X_train, y_train)
print("\nModel training complete.")

# Make predictions on the test set
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation Metrics:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2) Score: {r2:.4f}")


Training set shape: (28, 8)
Testing set shape: (8, 8)
['Year', 'Tobacco Price\nIndex', 'Retail Prices\nIndex', 'Tobacco Price Index Relative to Retail Price Index', "Real Households' Disposable Income", 'Affordability of Tobacco Index', 'Household Expenditure on Tobacco', 'Household Expenditure Total', 'Expenditure on Tobacco as a Percentage of Expenditure']
['Year', 'Tobacco Price\nIndex', 'Retail Prices\nIndex', 'Tobacco Price Index Relative to Retail Price Index', "Real Households' Disposable Income", 'Affordability of Tobacco Index', 'Household Expenditure on Tobacco', 'Household Expenditure Total']

--- Model Training and Evaluation ---

Model training complete.

Model Evaluation Metrics:
Mean Absolute Error (MAE): 0.0960
Mean Squared Error (MSE): 0.0151
Root Mean Squared Error (RMSE): 0.1230
R-squared (R2) Score: 0.8899
